# Spaceship Titanic — Passenger Transport Prediction

A machine learning classification project for the Kaggle **Spaceship Titanic** competition.

This notebook presents the complete modeling workflow: data preparation, feature engineering, model comparison, ensemble experiments, and final CatBoost tuning.

## 1. Problem Overview

The objective is to predict whether a passenger was transported to another dimension after the Spaceship Titanic anomaly.

**Target:** `Transported`

The workflow covers data loading, feature engineering, missing-value handling, categorical encoding, model comparison, ensemble strategies, and final hyperparameter tuning.

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

print("Libraries loaded successfully.")

Libraries loaded successfully.


## 2. Load the Dataset

In [2]:
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
display(train.head())

Train shape: (8693, 14)
Test shape : (4277, 13)


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [3]:
print("Missing values:")
display(train.isnull().sum().sort_values(ascending=False).head(20))

print("\nTarget distribution:")
display(train["Transported"].value_counts())

print("\nData types:")
display(train.dtypes)

Missing values:


CryoSleep       217
ShoppingMall    208
VIP             203
HomePlanet      201
Name            200
Cabin           199
VRDeck          188
Spa             183
FoodCourt       183
Destination     182
RoomService     181
Age             179
PassengerId       0
Transported       0
dtype: int64


Target distribution:


Transported
True     4378
False    4315
Name: count, dtype: int64


Data types:


PassengerId         str
HomePlanet          str
CryoSleep        object
Cabin               str
Destination         str
Age             float64
VIP              object
RoomService     float64
FoodCourt       float64
ShoppingMall    float64
Spa             float64
VRDeck          float64
Name                str
Transported        bool
dtype: object

## 3. Feature Engineering

Passenger and cabin identifiers contain useful structural information. Spending columns are also aggregated into a single behavioral feature.

In [4]:
def preprocess(df):
    df = df.copy()

    df["GroupId"] = df["PassengerId"].str.split("_").str[0]
    df["PersonNumber"] = df["PassengerId"].str.split("_").str[1].astype(int)

    cabin = df["Cabin"].str.split("/", expand=True)
    df["Deck"] = cabin[0]
    df["CabinNumber"] = pd.to_numeric(cabin[1], errors="coerce")
    df["Side"] = cabin[2]

    spending = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
    df["TotalSpent"] = df[spending].sum(axis=1)

    df["CryoSleep"] = df["CryoSleep"].astype("Int64")
    df["VIP"] = df["VIP"].astype("Int64")

    numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns
    for col in numeric_cols:
        df[col] = df[col].fillna(df[col].median())

    categorical_cols = df.select_dtypes(include=["object"]).columns
    for col in categorical_cols:
        df[col] = df[col].fillna("Missing")

    df["CryoSleep"] = df["CryoSleep"].fillna(0)
    df["VIP"] = df["VIP"].fillna(0)

    df = df.drop(["Name"], axis=1)

    return df

train_clean = preprocess(train)
test_clean = preprocess(test)

print("Train shape:", train_clean.shape)
print("Test shape :", test_clean.shape)
print("Remaining missing values:", train_clean.isnull().sum().sum())

Train shape: (8693, 19)
Test shape : (4277, 18)
Remaining missing values: 0


C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\1350240065.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=["object"]).columns
C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\1350240065.py:22: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/doc

## 4. Additional Features

Group size, solo-travel status, spending behavior, age groups, and cabin sections are added to provide the models with higher-level signals.

In [5]:
y = train_clean["Transported"].astype(int)
test_ids = test_clean["PassengerId"]

X = train_clean.drop("Transported", axis=1)
X_test = test_clean.copy()

group_counts = pd.concat([X["GroupId"], X_test["GroupId"]]).value_counts()

X["FamilySize"] = X["GroupId"].map(group_counts)
X_test["FamilySize"] = X_test["GroupId"].map(group_counts)

X["IsAlone"] = (X["FamilySize"] == 1).astype(int)
X_test["IsAlone"] = (X_test["FamilySize"] == 1).astype(int)

X["NoSpending"] = (X["TotalSpent"] == 0).astype(int)
X_test["NoSpending"] = (X_test["TotalSpent"] == 0).astype(int)

X["IsChild"] = (X["Age"] < 13).astype(int)
X_test["IsChild"] = (X_test["Age"] < 13).astype(int)

X["IsYoung"] = ((X["Age"] >= 13) & (X["Age"] < 25)).astype(int)
X_test["IsYoung"] = ((X_test["Age"] >= 13) & (X_test["Age"] < 25)).astype(int)

X["CabinSection"] = pd.cut(X["CabinNumber"], bins=[-1, 100, 200, 300, 400, 500, 1000], labels=False)
X_test["CabinSection"] = pd.cut(X_test["CabinNumber"], bins=[-1, 100, 200, 300, 400, 500, 1000], labels=False)

for col in X.select_dtypes(include="object").columns:
    X[col] = X[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

    le = LabelEncoder()
    le.fit(pd.concat([X[col], X_test[col]]))
    X[col] = le.transform(X[col])
    X_test[col] = le.transform(X_test[col])

X = X.fillna(X.median())
X_test = X_test.fillna(X.median())

X = X.drop(["PassengerId", "Cabin"], axis=1)
X_test = X_test.drop(["PassengerId", "Cabin"], axis=1)

print("Final training shape:", X.shape)
print("Final test shape:", X_test.shape)

Final training shape: (8693, 22)
Final test shape: (4277, 22)


C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\3265187222.py:27: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in X.select_dtypes(include="object").columns:


## 5. Train / Validation Split

In [6]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Validation samples:", len(X_val))

Training samples: 6954
Validation samples: 1739


## 6. Model Experiments

Multiple algorithms were tested during development. The goal was not only to obtain a strong score, but also to compare different modeling approaches.

In [7]:
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_val_pred = rf_model.predict(X_val)

print("Random Forest Accuracy:", accuracy_score(y_val, rf_val_pred))

Random Forest Accuracy: 0.8085106382978723


In [8]:
cat_train = train_clean.drop("Transported", axis=1).copy()
cat_test = test_clean.copy()
cat_cols = cat_train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    cat_train[col] = cat_train[col].astype(str)
    cat_test[col] = cat_test[col].astype(str)

cat_train, cat_val, cat_y_train, cat_y_val = train_test_split(
    cat_train, y, test_size=0.2, random_state=42, stratify=y
)

cat_model = CatBoostClassifier(
    iterations=800, learning_rate=0.05, depth=6,
    loss_function="Logloss", eval_metric="Accuracy",
    verbose=100, random_seed=42
)

cat_model.fit(cat_train, cat_y_train, cat_features=cat_cols)
cat_val_pred = cat_model.predict(cat_val).astype(int).ravel()

print("CatBoost Accuracy:", accuracy_score(cat_y_val, cat_val_pred))

C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\2701726387.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = cat_train.select_dtypes(include=["object"]).columns.tolist()


0:	learn: 0.7592752	total: 185ms	remaining: 2m 27s
100:	learn: 0.8183779	total: 2.31s	remaining: 16s
200:	learn: 0.8363532	total: 4.3s	remaining: 12.8s
300:	learn: 0.8498706	total: 6.37s	remaining: 10.6s
400:	learn: 0.8639632	total: 8.36s	remaining: 8.32s
500:	learn: 0.8731665	total: 10.4s	remaining: 6.19s
600:	learn: 0.8843831	total: 12.4s	remaining: 4.1s
700:	learn: 0.8920046	total: 14.3s	remaining: 2.03s
799:	learn: 0.8984757	total: 16.3s	remaining: 0us
CatBoost Accuracy: 0.8148361127084531


In [9]:
xgb_train = train_clean.drop("Transported", axis=1).copy()
xgb_test = test_clean.copy()

xgb_train = xgb_train.drop("PassengerId", axis=1)
xgb_test = xgb_test.drop("PassengerId", axis=1)

combined = pd.concat([xgb_train, xgb_test], axis=0)
combined = pd.get_dummies(combined, columns=combined.select_dtypes(include="object").columns)

xgb_train = combined.iloc[:len(xgb_train)].copy()
xgb_test = combined.iloc[len(xgb_train):].copy()

xgb_train = xgb_train.astype(float)
xgb_test = xgb_test.astype(float)
xgb_train = xgb_train.fillna(xgb_train.median())
xgb_test = xgb_test.fillna(xgb_train.median())

xgb_model = XGBClassifier(
    n_estimators=600, learning_rate=0.03, max_depth=5,
    subsample=0.8, colsample_bytree=0.8,
    objective="binary:logistic", eval_metric="logloss",
    random_state=42, n_jobs=-1
)

xgb_model.fit(xgb_train, y)
xgb_predictions = xgb_model.predict(xgb_test).astype(bool)

pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": xgb_predictions
}).to_csv("submission_xgboost.csv", index=False)

print("XGBoost submission created.")

C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\497834489.py:8: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  combined = pd.get_dummies(combined, columns=combined.select_dtypes(include="object").columns)


XGBoost submission created.


## 7. Neural Network and KNN

Neural Network and KNN were also tested as alternative approaches. Both require numerical representations of the categorical features and standardized inputs.

In [10]:
nn_train = train_clean.drop("Transported", axis=1).copy().drop("PassengerId", axis=1)
nn_test = test_clean.copy().drop("PassengerId", axis=1)

combined = pd.concat([nn_train, nn_test], axis=0)
combined = pd.get_dummies(combined, columns=combined.select_dtypes(include="object").columns)

nn_train = combined.iloc[:len(nn_train)].copy().astype(float)
nn_test = combined.iloc[len(nn_train):].copy().astype(float)

nn_train = nn_train.fillna(nn_train.median())
nn_test = nn_test.fillna(nn_train.median())

scaler = StandardScaler()
nn_train = scaler.fit_transform(nn_train)
nn_test = scaler.transform(nn_test)

print("Neural Network input shape:", nn_train.shape)

C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\1467652376.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  combined = pd.get_dummies(combined, columns=combined.select_dtypes(include="object").columns)


Neural Network input shape: (8693, 19137)


In [11]:
knn_train = train_clean.drop("Transported", axis=1).copy().drop("PassengerId", axis=1)
knn_test = test_clean.copy().drop("PassengerId", axis=1)

combined = pd.concat([knn_train, knn_test], axis=0)
combined = pd.get_dummies(combined, columns=combined.select_dtypes(include="object").columns)

knn_train = combined.iloc[:len(knn_train)].copy().astype(float)
knn_test = combined.iloc[len(knn_train):].copy().astype(float)

knn_train = knn_train.fillna(knn_train.median())
knn_test = knn_test.fillna(knn_train.median())

scaler = StandardScaler()
knn_train = scaler.fit_transform(knn_train)
knn_test = scaler.transform(knn_test)

knn_model = KNeighborsClassifier(n_neighbors=15, weights="distance", n_jobs=-1)
knn_model.fit(knn_train, y)
knn_predictions = knn_model.predict(knn_test)

pd.DataFrame({
    "PassengerId": test_ids,
    "Transported": knn_predictions
}).to_csv("submission_knn.csv", index=False)

print("KNN submission created.")

C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\3222933531.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  combined = pd.get_dummies(combined, columns=combined.select_dtypes(include="object").columns)


KNN submission created.


## 8. Model Comparison

The Kaggle submissions generated during experimentation were compared using their competition scores.

In [12]:
model_results = pd.DataFrame({
    "Model": [
        "Random Forest baseline",
        "Random Forest + feature engineering",
        "CatBoost",
        "XGBoost",
        "Neural Network",
        "KNN"
    ],
    "Kaggle Score": [0.79541, 0.79003, 0.80687, 0.80079, 0.62450, 0.70797]
})

display(model_results.sort_values("Kaggle Score", ascending=False).reset_index(drop=True))

,Model,Kaggle Score
0,CatBoost,0.80687
1,XGBoost,0.80079
2,Random Forest baseline,0.79541
3,Random Forest + feature engineering,0.79003
4,KNN,0.70797
5,Neural Network,0.62450


## 9. Ensemble Experiments

Predictions from six model submissions were combined. Two strategies were evaluated: score-weighted voting and a rule-based ensemble that prioritizes agreement between the strongest tree-based models.

In [14]:
files = [
    "submission1.csv",
    "submission2.csv",
    "submission_catboost.csv",
    "submission_Xgboost.csv",
    "submission_nn.csv",
    "submission_knn.csv"
]

scores = np.array([0.79541, 0.79003, 0.80687, 0.80079, 0.62450, 0.70797])

dfs = []

for i, file in enumerate(files):
    df = pd.read_csv(file).rename(columns={"Transported": f"Model_{i+1}"})
    dfs.append(df)

submission_db = dfs[0][["PassengerId", "Model_1"]].copy()

for i in range(1, 6):
    submission_db = submission_db.merge(
        dfs[i][["PassengerId", f"Model_{i+1}"]],
        on="PassengerId"
    )

submission_db.to_csv("submission_database.csv", index=False)

weights = scores / scores.sum()

weighted_prediction = np.average(
    submission_db[["Model_1", "Model_2", "Model_3", "Model_4", "Model_5", "Model_6"]].values,
    axis=1,
    weights=weights
)

submission_weighted = pd.DataFrame({
    "PassengerId": submission_db["PassengerId"],
    "Transported": weighted_prediction >= 0.5
})

submission_weighted.to_csv("submission_weighted_ensemble.csv", index=False)

print("Weighted ensemble submission created.")

Weighted ensemble submission created.


In [15]:
db = submission_db.copy()
models = ["Model_1", "Model_2", "Model_3", "Model_4", "Model_5", "Model_6"]

for col in models:
    db[col] = db[col].astype(int)

weights = scores / scores.sum()

final_prediction = []
condition_1 = condition_2 = condition_3 = 0

for _, row in db.iterrows():
    m1, m2, m3, m4, m5, m6 = [row[col] for col in models]

    if m3 == m4:
        prediction = m3
        condition_1 += 1
    else:
        first_four = [m1, m2, m3, m4]
        true_count = sum(first_four)
        false_count = 4 - true_count

        if true_count >= 3:
            prediction = 1
            condition_2 += 1
        elif false_count >= 3:
            prediction = 0
            condition_2 += 1
        else:
            prediction = int(np.average([m1, m2, m3, m4, m5, m6], weights=weights) >= 0.5)
            condition_3 += 1

    final_prediction.append(prediction)

submission_rule_based = pd.DataFrame({
    "PassengerId": db["PassengerId"],
    "Transported": np.array(final_prediction).astype(bool)
})

submission_rule_based.to_csv("submission_rule_based_ensemble.csv", index=False)

print("Total predictions:", len(submission_rule_based))
print("Condition 1:", condition_1)
print("Condition 2:", condition_2)
print("Condition 3:", condition_3)
print("Rule-based ensemble submission created.")

Total predictions: 4277
Condition 1: 4053
Condition 2: 192
Condition 3: 32
Rule-based ensemble submission created.


## 10. Final CatBoost Tuning

CatBoost was the strongest individual model in the experiments. A randomized search over 25 configurations was therefore used to improve the final model.

The search varies learning rate, tree depth, L2 regularization, random strength, bagging temperature, border count and feature sampling ratio. Early stopping selects the best iteration on the validation set.

In [17]:
train_final = pd.read_csv("train_clean.csv")
test_final = pd.read_csv("test_clean.csv")

y_final = train_final["Transported"].astype(int)
test_ids_final = test_final["PassengerId"]

X_final = train_final.drop("Transported", axis=1).drop("PassengerId", axis=1)
X_test_final = test_final.drop("PassengerId", axis=1)

cat_cols_final = X_final.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols_final:
    X_final[col] = X_final[col].astype(str)
    X_test_final[col] = X_test_final[col].astype(str)

X_train_final, X_val_final, y_train_final, y_val_final = train_test_split(
    X_final, y_final, test_size=0.2, random_state=42, stratify=y_final
)

rng = np.random.RandomState(42)

configs = []

for _ in range(25):
    configs.append({
        "iterations": 2000,
        "learning_rate": rng.choice([0.02, 0.025, 0.03, 0.04, 0.05]),
        "depth": rng.choice([5, 6, 7, 8]),
        "l2_leaf_reg": rng.choice([3, 5, 7, 10, 15]),
        "random_strength": rng.choice([0.2, 0.5, 1, 1.5, 2]),
        "bagging_temperature": rng.choice([0, 0.5, 1, 2]),
        "border_count": rng.choice([64, 128, 254]),
        "rsm": rng.choice([0.7, 0.8, 0.9, 1.0])
    })

results = []

for i, params in enumerate(configs):
    model = CatBoostClassifier(
        **params,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=42,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1
    )

    model.fit(
        X_train_final, y_train_final,
        cat_features=cat_cols_final,
        eval_set=(X_val_final, y_val_final),
        use_best_model=True,
        early_stopping_rounds=100,
        verbose=False
    )

    score = model.get_best_score()["validation"]["Accuracy"]
    best_iteration = model.get_best_iteration()

    results.append({
        "score": score,
        "iteration": best_iteration,
        "params": params
    })

    print(f"{i+1:02d}/25 | Accuracy: {score:.6f} | Best Iteration: {best_iteration}")

results = sorted(results, key=lambda x: x["score"], reverse=True)
best = results[0]

print("\nBEST VALIDATION RESULT")
print("Accuracy:", best["score"])
print("Best Iteration:", best["iteration"])
print("Parameters:")
print(best["params"])

C:\Users\Danial\AppData\Local\Temp\ipykernel_14696\2885261742.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols_final = X_final.select_dtypes(include=["object"]).columns.tolist()


01/25 | Accuracy: 0.810811 | Best Iteration: 559
02/25 | Accuracy: 0.815411 | Best Iteration: 408
03/25 | Accuracy: 0.824037 | Best Iteration: 816
04/25 | Accuracy: 0.798160 | Best Iteration: 117
05/25 | Accuracy: 0.821737 | Best Iteration: 335
06/25 | Accuracy: 0.807936 | Best Iteration: 231
07/25 | Accuracy: 0.802185 | Best Iteration: 24
08/25 | Accuracy: 0.811386 | Best Iteration: 437
09/25 | Accuracy: 0.801035 | Best Iteration: 410
10/25 | Accuracy: 0.804485 | Best Iteration: 213
11/25 | Accuracy: 0.822887 | Best Iteration: 608
12/25 | Accuracy: 0.797585 | Best Iteration: 106
13/25 | Accuracy: 0.814261 | Best Iteration: 487
14/25 | Accuracy: 0.802185 | Best Iteration: 420
15/25 | Accuracy: 0.815411 | Best Iteration: 440
16/25 | Accuracy: 0.820012 | Best Iteration: 671
17/25 | Accuracy: 0.802760 | Best Iteration: 97
18/25 | Accuracy: 0.821737 | Best Iteration: 511
19/25 | Accuracy: 0.825187 | Best Iteration: 546
20/25 | Accuracy: 0.820587 | Best Iteration: 399
21/25 | Accuracy: 0.81

## 11. Final Model and Submission

In [18]:
final_params = best["params"].copy()
final_params["iterations"] = best["iteration"] + 1

final_model = CatBoostClassifier(
    **final_params,
    loss_function="Logloss",
    eval_metric="Accuracy",
    random_seed=42,
    verbose=200,
    allow_writing_files=False,
    thread_count=-1
)

final_model.fit(X_final, y_final, cat_features=cat_cols_final)

final_predictions = final_model.predict(X_test_final).astype(bool).ravel()

submission_final = pd.DataFrame({
    "PassengerId": test_ids_final,
    "Transported": final_predictions
})

submission_final.to_csv("submission_final_catboost.csv", index=False)

print("\nFINAL SUBMISSION CREATED")
display(submission_final.head())
print("\nFile: submission_final_catboost.csv")

0:	learn: 0.7519844	total: 20ms	remaining: 10.9s
200:	learn: 0.8372254	total: 4.21s	remaining: 7.24s
400:	learn: 0.8605775	total: 8.26s	remaining: 3.01s
546:	learn: 0.8746118	total: 11.3s	remaining: 0us

FINAL SUBMISSION CREATED


,PassengerId,Transported
0,0013_01,True
1,0018_01,False
2,0019_01,True
3,0021_01,True
4,0023_01,True



File: submission_final_catboost.csv


## 12. Final Result

The final tuned CatBoost model achieved the best competition result from the project:

**Kaggle Score: 0.80757**

### Key Takeaways

- Passenger and cabin identifiers contain useful structural information.
- Spending behavior can be summarized into informative aggregate features.
- CatBoost performed strongly on the mixed numerical/categorical feature space.
- XGBoost provided a competitive alternative.
- Neural Network and KNN were tested but performed substantially worse.
- Multiple ensemble strategies were explored before final CatBoost tuning.
- The final tuned CatBoost model was selected as the project's final submission.